
# Consumption–Saving RL — Discrete PPO

이번 실험은 PPO를 DQN과 **동일한 20개 discrete saving-share actions**로 학습합니다.

\[
z \in \{0, 0.05, 0.10, \ldots, 0.95\}
\]

왜 이렇게 바꾸는가?

1. 기존 continuous PPO는 Gaussian action + clipping 때문에 boundary collapse가 발생했습니다.
2. Discrete PPO는 Categorical policy를 사용하므로 그 문제가 없습니다.
3. DQN과 PPO가 **완전히 같은 state/action/reward 문제**를 풀게 되어 알고리즘 비교가 공정해집니다.
4. benchmark는 DP-Matched를 사용합니다.

이번에는 100k timesteps 동안 10k마다 validation을 하고 best checkpoint를 저장합니다.
`Run all`에서는 heavy training이 자동 실행되지 않습니다.


In [ ]:

# ============================================================
# 0. Runtime setup
# ============================================================

import sys
import subprocess
import importlib
import os

try:
    import gymnasium as gym
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "gymnasium"]
    )
    import gymnasium as gym

import sympy
sympy_printing = importlib.import_module("sympy.printing")
if not hasattr(sympy, "printing"):
    sympy.printing = sympy_printing

try:
    import stable_baselines3
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "stable-baselines3"]
    )
    import stable_baselines3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gymnasium import spaces
import torch

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)
print("SB3:", stable_baselines3.__version__)


In [ ]:

# ============================================================
# 1. Calibration
# ============================================================

R = 1.00985
BETA = 0.9703
SIGMA = 2.0
Y_HIGH = 1.0
Y_LOW = 0.472

P_INCOME = np.array(
    [
        [0.9393, 0.0607],
        [0.3920, 0.6080]
    ],
    dtype=np.float64
)

SAVING_SHARES = np.arange(
    0.0,
    1.0,
    0.05,
    dtype=np.float64
)

A_MAX = 4.5
N_ASSETS = 500


In [ ]:

# ============================================================
# 2. Discrete consumption-saving environment
# ============================================================

class DiscreteConsumptionSavingEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(
        self,
        random_initial_state=True,
        initial_asset=1.0,
        initial_income_state=0
    ):
        super().__init__()

        self.random_initial_state = random_initial_state
        self.initial_asset = float(initial_asset)
        self.initial_income_state = int(initial_income_state)

        self.action_space = spaces.Discrete(
            len(SAVING_SHARES)
        )

        # observation = [asset / 4.5, income]
        self.observation_space = spaces.Box(
            low=np.array(
                [0.0, Y_LOW],
                dtype=np.float32
            ),
            high=np.array(
                [np.inf, Y_HIGH],
                dtype=np.float32
            ),
            dtype=np.float32
        )

        self.asset = None
        self.income_state = None

    def _obs(self):
        return np.array(
            [
                self.asset / 4.5,
                Y_HIGH if self.income_state == 0 else Y_LOW
            ],
            dtype=np.float32
        )

    def _utility(self, c):
        if np.isclose(SIGMA, 1.0):
            return float(np.log(c))

        return float(
            c ** (1.0 - SIGMA)
            / (1.0 - SIGMA)
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        options = options or {}

        if "initial_asset" in options:
            self.asset = float(options["initial_asset"])
        elif self.random_initial_state:
            self.asset = float(
                self.np_random.uniform(0.0, 4.5)
            )
        else:
            self.asset = self.initial_asset

        if "initial_income_state" in options:
            self.income_state = int(
                options["initial_income_state"]
            )
        elif self.random_initial_state:
            self.income_state = int(
                self.np_random.integers(0, 2)
            )
        else:
            self.income_state = self.initial_income_state

        return self._obs(), {}

    def step(self, action):

        z = float(
            SAVING_SHARES[int(action)]
        )

        income = (
            Y_HIGH if self.income_state == 0 else Y_LOW
        )

        cash_on_hand = (
            R * self.asset + income
        )

        next_asset = (
            z * cash_on_hand
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = self._utility(
            consumption
        )

        next_income_state = int(
            self.np_random.choice(
                2,
                p=P_INCOME[self.income_state]
            )
        )

        self.asset = next_asset
        self.income_state = next_income_state

        return (
            self._obs(),
            reward,
            False,
            False,
            {"saving_share": z}
        )


def make_train_env():

    env = DiscreteConsumptionSavingEnv(
        random_initial_state=True
    )

    # computational truncation only
    env = gym.wrappers.TimeLimit(
        env,
        max_episode_steps=200
    )

    return Monitor(env)


print("Discrete PPO environment ready.")


In [ ]:

# ============================================================
# 3. DP-Matched benchmark
# ============================================================

def solve_dp_matched(
    tolerance=1e-10,
    max_iterations=5000
):

    asset_grid = np.linspace(
        0.0,
        A_MAX,
        N_ASSETS
    )

    V = np.zeros(
        (N_ASSETS, 2),
        dtype=np.float64
    )

    policy_share = np.zeros_like(V)

    for iteration in range(max_iterations):

        V_new = np.empty_like(V)

        for income_state, income in [
            (0, Y_HIGH),
            (1, Y_LOW)
        ]:

            cash_on_hand = (
                R * asset_grid
                + income
            )

            Q = np.full(
                (N_ASSETS, len(SAVING_SHARES)),
                -np.inf,
                dtype=np.float64
            )

            for j, z in enumerate(SAVING_SHARES):

                next_asset = (
                    z * cash_on_hand
                )

                consumption = (
                    (1.0 - z)
                    * cash_on_hand
                )

                feasible = (
                    (consumption > 1e-12)
                    &
                    (next_asset <= A_MAX)
                )

                V_H = np.interp(
                    next_asset,
                    asset_grid,
                    V[:, 0]
                )

                V_L = np.interp(
                    next_asset,
                    asset_grid,
                    V[:, 1]
                )

                expected_future = (
                    P_INCOME[income_state, 0] * V_H
                    +
                    P_INCOME[income_state, 1] * V_L
                )

                if np.isclose(SIGMA, 1.0):
                    utility = np.log(consumption)
                else:
                    utility = (
                        consumption ** (1.0 - SIGMA)
                        /
                        (1.0 - SIGMA)
                    )

                q = (
                    utility
                    +
                    BETA * expected_future
                )

                q[~feasible] = -np.inf
                Q[:, j] = q

            best_idx = np.argmax(
                Q,
                axis=1
            )

            V_new[:, income_state] = (
                Q[
                    np.arange(N_ASSETS),
                    best_idx
                ]
            )

            policy_share[:, income_state] = (
                SAVING_SHARES[best_idx]
            )

        diff = np.max(
            np.abs(V_new - V)
        )

        V = V_new

        if diff < tolerance:
            break

    return asset_grid, V, policy_share


asset_grid_m, V_dp_matched, policy_share_matched = (
    solve_dp_matched()
)

dp_matched_value = np.interp(
    1.0,
    asset_grid_m,
    V_dp_matched[:, 0]
)

print(
    "DP-Matched Bellman value:",
    dp_matched_value
)


In [ ]:

# ============================================================
# 4. Evaluation helpers
# ============================================================

def utility_vectorized(c):

    if np.isclose(SIGMA, 1.0):
        return np.log(c)

    return (
        c ** (1.0 - SIGMA)
        /
        (1.0 - SIGMA)
    )


def evaluate_discrete_ppo(
    model,
    n_episodes=2000,
    horizon=500,
    seed=10000
):

    rng = np.random.default_rng(seed)

    assets = np.ones(
        n_episodes,
        dtype=np.float64
    )

    income_states = np.zeros(
        n_episodes,
        dtype=int
    )

    total_utility = np.zeros(
        n_episodes,
        dtype=np.float64
    )

    discount = 1.0

    for _ in range(horizon):

        incomes = np.where(
            income_states == 0,
            Y_HIGH,
            Y_LOW
        )

        obs = np.column_stack(
            [
                assets / 4.5,
                incomes
            ]
        ).astype(np.float32)

        actions, _ = model.predict(
            obs,
            deterministic=True
        )

        actions = np.asarray(
            actions
        ).reshape(-1).astype(int)

        z = SAVING_SHARES[actions]

        cash_on_hand = (
            R * assets
            + incomes
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = utility_vectorized(
            consumption
        )

        total_utility += (
            discount * reward
        )

        assets = (
            z * cash_on_hand
        )

        p_low = np.where(
            income_states == 0,
            P_INCOME[0, 1],
            P_INCOME[1, 1]
        )

        income_states = (
            rng.random(n_episodes)
            < p_low
        ).astype(int)

        discount *= BETA

    mean = total_utility.mean()
    std = total_utility.std(ddof=1)
    se = std / np.sqrt(n_episodes)
    ci = 1.96 * se

    return {
        "mean_utility": mean,
        "std_utility": std,
        "ci_lower": mean - ci,
        "ci_upper": mean + ci,
        "max_final_asset": assets.max()
    }


def get_discrete_ppo_policy(
    model,
    asset_grid
):

    policy = np.zeros(
        (len(asset_grid), 2)
    )

    for income_state, income in [
        (0, Y_HIGH),
        (1, Y_LOW)
    ]:

        obs = np.column_stack(
            [
                asset_grid / 4.5,
                np.full(
                    len(asset_grid),
                    income
                )
            ]
        ).astype(np.float32)

        actions, _ = model.predict(
            obs,
            deterministic=True
        )

        actions = np.asarray(
            actions
        ).reshape(-1).astype(int)

        policy[:, income_state] = (
            SAVING_SHARES[actions]
        )

    return policy



## Heavy training cell

**계산부하: 무거움.**

- PPO 총 100k timesteps
- 10k마다 validation
- seed=42 하나만 사용
- 이전 continuous PPO 50k보다 총 학습량은 2배

`Run all`에서는 자동 실행되지 않습니다.


In [ ]:

RUN_HEAVY_TRAINING = False

if not RUN_HEAVY_TRAINING:

    print(
        "Fast setup complete.\n"
        "Change RUN_HEAVY_TRAINING = True and rerun THIS CELL ONLY."
    )

else:

    train_env = make_train_env()

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        gamma=BETA,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,
        verbose=0,
        seed=42,
        device="auto"
    )

    checkpoint_rows = []

    best_val = -np.inf
    best_step = None

    best_path = "/tmp/best_discrete_ppo"

    # 10 chunks x 10k = 100k total
    for step in range(
        10_000,
        100_001,
        10_000
    ):

        model.learn(
            total_timesteps=10_000,
            reset_num_timesteps=False
        )

        val = evaluate_discrete_ppo(
            model,
            n_episodes=512,
            horizon=300,
            seed=54321
        )

        policy = get_discrete_ppo_policy(
            model,
            asset_grid_m
        )

        err = np.abs(
            policy
            - policy_share_matched
        )

        row = {
            "step": step,
            "validation_utility":
                val["mean_utility"],
            "policy_mae":
                err.mean(),
            "exact":
                np.mean(
                    np.isclose(
                        policy,
                        policy_share_matched
                    )
                ),
            "within5":
                np.mean(
                    err <= 0.0500001
                ),
            "within10":
                np.mean(
                    err <= 0.1000001
                )
        }

        checkpoint_rows.append(row)

        if val["mean_utility"] > best_val:
            best_val = val["mean_utility"]
            best_step = step
            model.save(best_path)

        print(
            f"step={step:6d} | "
            f"val={val['mean_utility']:.4f} | "
            f"MAE={err.mean():.4f} | "
            f"best={best_step}"
        )

    history = pd.DataFrame(
        checkpoint_rows
    )

    # final model test
    final_test = evaluate_discrete_ppo(
        model,
        n_episodes=2000,
        horizon=500,
        seed=10000
    )

    final_policy = get_discrete_ppo_policy(
        model,
        asset_grid_m
    )

    final_err = np.abs(
        final_policy
        - policy_share_matched
    )

    # best checkpoint test
    best_model = PPO.load(
        best_path,
        env=make_train_env()
    )

    best_test = evaluate_discrete_ppo(
        best_model,
        n_episodes=2000,
        horizon=500,
        seed=10000
    )

    best_policy = get_discrete_ppo_policy(
        best_model,
        asset_grid_m
    )

    best_err = np.abs(
        best_policy
        - policy_share_matched
    )

    print("\nBest discrete PPO checkpoint:", best_step)

    comparison = pd.DataFrame(
        [
            {
                "Model": "DP-Matched MC",
                "Mean Utility": -36.254708,
                "Policy MAE": 0.0
            },
            {
                "Model": "Best DQN checkpoint",
                "Mean Utility": -36.492848,
                "Policy MAE": 0.05535
            },
            {
                "Model": "Discrete PPO final",
                "Mean Utility":
                    final_test["mean_utility"],
                "Policy MAE":
                    final_err.mean()
            },
            {
                "Model":
                    f"Discrete PPO best @{best_step}",
                "Mean Utility":
                    best_test["mean_utility"],
                "Policy MAE":
                    best_err.mean()
            }
        ]
    )

    display(comparison)

    print(
        "\nBest PPO 95% CI:",
        (
            best_test["ci_lower"],
            best_test["ci_upper"]
        )
    )

    print(
        "Exact:",
        np.mean(
            np.isclose(
                best_policy,
                policy_share_matched
            )
        )
    )

    print(
        "Within 5pp:",
        np.mean(
            best_err <= 0.0500001
        )
    )

    print(
        "Within 10pp:",
        np.mean(
            best_err <= 0.1000001
        )
    )

    # validation curve
    plt.figure(figsize=(10, 5))

    plt.plot(
        history["step"],
        history["validation_utility"],
        marker="o"
    )

    plt.axvline(
        best_step,
        linestyle="--",
        label=f"Best = {best_step}"
    )

    plt.xlabel("Training Step")
    plt.ylabel("Validation Discounted Utility")
    plt.title("Discrete PPO Validation Performance")
    plt.legend()
    plt.show()

    # policy plots
    plt.figure(figsize=(10, 5))

    plt.plot(
        asset_grid_m,
        policy_share_matched[:, 0],
        label="DP-Matched: High"
    )

    plt.plot(
        asset_grid_m,
        best_policy[:, 0],
        "--",
        label=f"Discrete PPO best @{best_step}"
    )

    plt.xlabel("Current Asset")
    plt.ylabel("Saving Share")
    plt.title("Discrete PPO vs DP: High Income")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))

    plt.plot(
        asset_grid_m,
        policy_share_matched[:, 1],
        label="DP-Matched: Low"
    )

    plt.plot(
        asset_grid_m,
        best_policy[:, 1],
        "--",
        label=f"Discrete PPO best @{best_step}"
    )

    plt.xlabel("Current Asset")
    plt.ylabel("Saving Share")
    plt.title("Discrete PPO vs DP: Low Income")
    plt.legend()
    plt.show()
